<a href="https://colab.research.google.com/github/shruthilakshmi008-BA/ba-automation-suite/blob/main/01.%20Status-cleanser/Status_Cleanser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Installing OpenAI
!pip install openai

In [2]:
#File upload
from google.colab import files
uploaded = files.upload()

Saving messy_claims.csv to messy_claims.csv


In [3]:
#Printing the file path
import os
print(os.listdir())

['.config', 'messy_claims.csv', 'sample_data']


In [4]:
#Storing the OpenAI key securely in colab
from getpass import getpass

OPENAI_API_KEY = getpass(
    "Enter your OpenAI API key: "
)

Enter your OpenAI API key: ··········


In [5]:
#Getting the status by rules/AI
import pandas as pd
from openai import OpenAI

client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("OpenAI client created successfully")

CANONICAL_STATUSES = [
    "approved",
    "pending",
    "rejected"
]

RULES = {
    "approved": "approved",
    "appr": "approved",

    "pending review": "pending",
    "pending_review": "pending",
    "in review": "pending",
}

RULES = {
    "approved": "approved",
    "appr": "approved",
    "pending review": "pending",
    "pending_review": "pending",
    "in review": "pending",
}

##For overriding the openai failure
MOCK_AI_MAP = {
    "accepted": "approved",
    "waiting": "pending",
    "denied": "rejected",
    "claim declined": "rejected",
    "waiting for decision": "pending"
}

## Clean by Rules function
def clean_with_rules(raw_value: str):

    normalized = (
        raw_value
        .strip()
        .lower()
        .replace("_", " ")
    )
    return RULES.get(normalized)

#Cleaning with AI function
def clean_with_ai(raw_value: str, client=None) -> str:
    normalized = raw_value.strip().lower()

    # 1. Try OpenAI API call if client is available
    if client:
        try:
            print(f"[AI fallback] Classifying: '{raw_value}'")
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "system",
                        "content": "You classify claim statuses into exactly one word: approved, pending, rejected."
                    },
                    {
                        "role": "user",
                        "content": f"Classify this claim status: {raw_value}"
                    }
                ]
            )
            result = response.choices[0].message.content.strip().lower()

            # Validate AI response against allowed list
            if result in CANONICAL_STATUSES:
                return result
            else:
                print(f"WARNING: Unexpected AI response: {result}")

        except Exception as error:
            print(f"AI Error: {error}")

    # 2. Local Fallback (runs if client is None, API fails, or AI gives unexpected output)
    print(f"[Mock Fallback] Using lookup map for: '{raw_value}'")
    return MOCK_AI_MAP.get(normalized, "pending")


OpenAI client created successfully


In [6]:
def process_file(
    input_csv: str,
    output_csv: str
):

    print("Reading CSV file...")

    df = pd.read_csv(
        input_csv
    )

    cleaned_statuses = []
    methods_used = []

    for raw_value in df["status"]:

        # Convert value to string
        raw_value = str(raw_value)

        # Try rules first
        rule_result = clean_with_rules(
            raw_value
        )

        if rule_result:
            cleaned_statuses.append(
                rule_result
            )
            methods_used.append(
                "rule"
            )

        else:
            ai_result = clean_with_ai(
                raw_value
            )
            cleaned_statuses.append(
                ai_result
            )
            methods_used.append(
                "ai"
            )


    # Add cleaned status column
    df["status_cleaned"] = (
        cleaned_statuses
    )

    # Add method column
    df["method_used"] = (
        methods_used
    )

    # Save file
    df.to_csv(
        output_csv,
        index=False
    )

    # Count methods
    rule_count = methods_used.count(
        "rule"
    )
    ai_count = methods_used.count(
        "ai"
    )
    print("\nPROCESS COMPLETE")
    print(
        f"Rule matches: {rule_count}"
    )
    print(
        f"AI classifications: {ai_count}"
    )
    print(
        f"Saved file: {output_csv}"
    )

In [7]:
process_file(
    "messy_claims.csv",
    "cleaned_claims.csv"
)

Reading CSV file...
[Mock Fallback] Using lookup map for: 'accepted'
[Mock Fallback] Using lookup map for: 'waiting'
[Mock Fallback] Using lookup map for: 'denied'
[Mock Fallback] Using lookup map for: 'Claim Declined'
[Mock Fallback] Using lookup map for: 'Waiting for decision'

PROCESS COMPLETE
Rule matches: 7
AI classifications: 5
Saved file: cleaned_claims.csv
